# 🎯 [MOV01_M04] 네트워크 흐름(Max-Flow) 알고리즘 기반 병목 발생 지점 사전 식별 및 우회로 탐색 모델 고도화 (정답 솔루션)
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV01/mov01_m04_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 3) |
| **미션 ID** | MOV01_M04 |
| **미션 주제** | 네트워크 흐름(Max-Flow) 알고리즘 기반 병목 발생 지점 사전 식별 및 우회로 탐색 모델 고도화 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
수송 소요를 데이터로 예측해 군수 수송계획을 세우는 일은 구간별 통과 용량과 병목을 함께 보아야 정확해진다. 본 프로젝트는 수송 소요예측에서 예측 대상 정의, 시계열·피처 구성, 기준모델 비교, 설명가능성, 업무 임계값, 실적 대조를 익히도록 교육용 가상 노선망 자료를 코드로 만들어 구간별 수송 소요와 지체를 예측한다. 학습자는 병목 데이터 정의, 지체 전처리, BPR 기준 모델, 최대유량 기반 병목 식별과 피처 모델, 우회 소요·유류 산출과 앙상블 모델, 혼잡도 대시보드와 요인 설명, 작전군수관 승인(HITL)과 선형계획 최적화, 3개 구간 동시 차단 스트레스 시험, AAR 재보정과 ROI, 재학습 파이프라인 통합을 차례로 수행한다. 최종 목표는 구간별 소요·지체를 예측해 병목을 사전에 찾고 우회 물량 배분안을 만들되, 우회 명령의 하달 여부는 반드시 사람이 결정하는 수송 소요예측 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 야전(작전군수처, 수송대대)에 즉시 도입 가능한 "완성형 국방 AI 수송 시뮬레이터 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 기술 명세 및 운영 매뉴얼** | MSR 통과 능력 데이터 명세서(`docs/mov01_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mov01_operations_guide.md`) | 미션 1, 10 |
| **`src/`** | **군수 데이터 정제 및 분석 모듈** | 통과 지체시간 계산 전처리 파이프라인(`src/data_preprocessing.py`), BPR 기반 1차 지체 예측 Baseline(`src/baseline_delay.py`) | 미션 2, 3 |
| **`notebooks/`** | **통계 검증 및 심층 EDA** | MSR 구간 지체 탐색적 데이터 분석 및 파레토 분석(`notebooks/mov01_eda_analysis.ipynb`) | 미션 3 |
| **`src/`** | **AI 예측 & 우회로 탐색 엔진** | Max-Flow 기반 병목 사전 식별 및 우회로 탐색(`src/network_flow.py`), 유류 소모 및 소요시간 산출(`src/route_cost_engine.py`) | 미션 4, 5 |
| **`src/`** | **작전 상황판 & 스트레스 모의** | Streamlit 기반 전술 COP 대시보드(`src/dashboard_app.py`), 3개 축선 봉쇄 스트레스 시험(`src/stress_test.py`) | 미션 6, 8 |
| **`pipeline/`** | **HITL 통제 & 지속적 학습** | 작전군수관 3단계 승인 파이프라인(`pipeline/hitl_workflow.py`), 챔피언/챌린저 모델 자동 재학습 | 미션 7, 10 |
| **`reports/`** | **검증 및 AAR 리포트** | 3개 축선 봉쇄 스트레스 검증 보고서(`reports/mov01_stress_test_report.md`), AAR 오차 분석 및 ROI 보고서(`reports/mov01_roi_analysis.md`) | 미션 8, 9 |
| **`root`** | **통합 배포 패키지** | 통합 실행 진입점(`main.py`), 국방 망분리 Dockerfile 패키지 | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 04. 네트워크 흐름(Max-Flow) 알고리즘 기반 병목 발생 지점 사전 식별 및 우회로 탐색 모델 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

네트워크 흐름 알고리즘 기반 병목 사전 식별 및 우회로 탐색 모델을 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **최대유량(Max-Flow) 알고리즘으로 병목 구간을 사전에 식별하고 구간 폐쇄 시 우회로를 찾으며, 파생 피처 기반 기준 모델을 구축한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **노선망에서 출발지에서 목적지까지의 최대 통과능력과 최소 절단 구간을 구하고 구간별 증설 민감도를 비교 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/features/mov01_feature_engineering.py 및 기준 모델` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 실무에서 납득 가능한 피처 설계의 논리성

#### 미션 PASS 조건

- 정량 기준: 파생 피처 10개 이상 생성, 기준 모델 MAPE 25% 이하 달성


- 진행 기준: 피처 파이프라인 및 베이스라인 모델 성능 지표 도출


- 검증 방법: 교차 검증(Time-series Cross Validation)을 통한 과적합 방지


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""최대유량(Max-Flow) 알고리즘으로 가상 노선망의 최소 절단(병목) 구간을 사전 식별하고 구간 폐쇄 시 우회 경로를 찾으며, 파생 피처 14종과 시계열 교차검증으로 지체 예측 기준 모델(MAPE)을 만든다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 노선망·구간 통과량 생성기 (실제 도로·지명·부대 없음)
# 가상 출발지(SRC)에서 가상 목적지(DST)까지 3개 축선(A·B·C)과 연결로 2개, 구간 10개. 150일치 피크시간 통과량·지체 시간을 만든다.
# ---------------------------------------------------------------------------
SEGS = pd.DataFrame([
    ("A1", "SRC", "J1", 12, 130, "일반"), ("A2", "J1", "J2", 8, 70, "교량"), ("A3", "J2", "DST", 10, 140, "교차로"),
    ("B1", "SRC", "J3", 15, 100, "일반"), ("B2", "J3", "J2", 9, 80, "터널"), ("B3", "J3", "DST", 14, 95, "일반"),
    ("C1", "SRC", "J4", 18, 70, "일반"), ("C2", "J4", "DST", 20, 75, "교량"),
    ("X1", "J1", "J3", 7, 50, "교차로"), ("X2", "J4", "J3", 10, 60, "일반")],
    columns=["seg", "u", "v", "km", "cap", "kind"])               # cap = 피크시간 통과 용량(대/시, 교육용 가정값)
KSPD = {"일반": 55, "교량": 40, "터널": 45, "교차로": 45}           # 구간 유형별 자유 주행 속도(km/h, 가정값)
SEGS["t0"] = SEGS["km"] / SEGS["kind"].map(KSPD) * 60              # 자유 주행 시간(분)
UTIL = dict(A1=.78, A2=.85, A3=.66, B1=.70, B2=.72, B3=.64, C1=.55, C2=.58, X1=.45, X2=.40)   # 평시 이용률


def bpr(t0, v, cap, a=0.15, b=4.0):
    """BPR 지체 함수(교통공학의 표준형): 통과량/용량 비가 커질수록 지체가 급증한다. (분)"""
    return t0 * a * (v / cap) ** b


def make_data(seed=11, days=150):
    """구간 x 일자 패널. v=실제 피크 통과량, plan_v=수송계획 수량(긴급 불출 미반영), delay=실제 지체(분)."""
    rng = np.random.default_rng(seed)
    d_idx = np.arange(days)
    lvl = np.array([1.0, 1.08, 1.04, 1.10, 1.02, 0.82, 0.78])[d_idx % 7] * (1 + 0.0012 * d_idx)
    lvl = lvl * np.where((d_idx >= 100) & (d_idx < 110), 1.30, 1.0)       # 수송량 폭증 기간(가상)
    prev, rows = {s: 0.0 for s in SEGS["seg"]}, []
    for d in d_idx:
        rain = bool(rng.random() < 0.18)
        for r in SEGS.itertuples():
            maint, urgent = bool(rng.random() < 0.06), bool(rng.random() < 0.05)   # 비계획 정비 / 긴급 불출
            v = r.cap * UTIL[r.seg] * lvl[d] * (1.25 if urgent else 1.0) * rng.normal(1, 0.05)
            cap_eff = r.cap * (0.65 if maint else 1.0) * (0.92 if rain else 1.0)
            delay = max(r.t0 * 0.2 * (v / cap_eff) ** 3.5 + 0.25 * prev[r.seg] + rng.normal(0, 0.4), 0.0)
            prev[r.seg] = delay
            rows.append(dict(day=int(d), dow=int(d % 7), seg=r.seg, kind=r.kind, km=r.km, cap=r.cap, t0=r.t0,
                             plan_v=v / (1.25 if urgent else 1.0) * rng.normal(1, 0.04), v=v, maint=int(maint), urgent=int(urgent),
                             rain=int(rain), cap_eff=cap_eff, delay=delay))
    df = pd.DataFrame(rows)
    df["transit"] = df["t0"] * (1 + 0.08 * df["rain"]) + df["delay"]
    return df


def mape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.mean(np.abs(y - p) / np.abs(y)) * 100)


def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(p, float))))
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# 지체 예측용 파생 피처 생성(당일 오전 시점에 알 수 있는 정보만 사용: 계획 물량, 공지 정비, 기상 예보, 전일·전주 실적)
# ---------------------------------------------------------------------------
FEATS = ["plan_v", "cap", "cap_eff", "t0", "km", "maint", "rain", "util", "util_eff", "util_eff4", "bpr_base", "lag1_delay", "lag7_delay",
         "roll7_v", "surge_ratio", "is_weekend", "dow_sin", "dow_cos", "k_교량", "k_터널", "k_교차로"]


def add_features(df):
    d = df.sort_values(["seg", "day"]).copy()
    g = d.groupby("seg")
    d["lag1_delay"] = g["delay"].shift(1)
    d["lag7_delay"] = g["delay"].shift(7)
    d["roll7_v"] = g["v"].transform(lambda s: s.shift(1).rolling(7).mean())
    d["util"] = d["plan_v"] / d["cap"]
    d["util_eff"] = d["plan_v"] / d["cap_eff"]
    d["util_eff4"] = d["util_eff"] ** 4
    d["bpr_base"] = bpr(d["t0"], d["plan_v"], d["cap"])
    d["surge_ratio"] = d["plan_v"] / d["roll7_v"]
    d["is_weekend"] = (d["dow"] >= 5).astype(int)
    d["dow_sin"], d["dow_cos"] = np.sin(2 * np.pi * d["dow"] / 7), np.cos(2 * np.pi * d["dow"] / 7)
    for k in ("교량", "터널", "교차로"):
        d["k_" + k] = (d["kind"] == k).astype(int)
    return d.dropna().sort_values(["day", "seg"]).reset_index(drop=True)
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# 노선망 흐름 도구: 최대유량(Edmonds-Karp), 증분 배정(혼잡 반영 최단시간 경로), 경로 열거
# ---------------------------------------------------------------------------
CAP = dict(zip(SEGS["seg"], SEGS["cap"]))
T0 = dict(zip(SEGS["seg"], SEGS["t0"]))
KM = dict(zip(SEGS["seg"], SEGS["km"]))
ARCS = {r.seg: (r.u, r.v) for r in SEGS.itertuples()}
SRC, DST = "SRC", "DST"


def seg_time(s, flow, cap=None, a=0.2, b=3.5):
    """구간 통과시간(분) = 자유주행 + 혼잡 지체. cap을 주면 용량을 대체(정비·기상 반영)."""
    r = flow / (cap or CAP[s])
    return T0[s] * (1 + a * r ** b + 6.0 * max(0.0, r - 1.0))        # 용량 초과분은 대기열로 급증


def max_flow(closed=(), capmul=None):
    """최대유량(대/시)과 최소 절단 구간(병목). capmul = {구간: 용량 배율}."""
    cap = {s: (0 if s in closed else CAP[s] * (capmul or {}).get(s, 1.0)) for s in CAP}
    res = {}
    for s, (u, v) in ARCS.items():
        res[(u, v)] = res.get((u, v), 0) + cap[s]
        res.setdefault((v, u), 0)
    flow = 0.0
    while True:
        prev, q = {SRC: None}, [SRC]
        for u in q:
            for (a, b), c in res.items():
                if a == u and c > 1e-9 and b not in prev:
                    prev[b] = u; q.append(b)
        if DST not in prev:
            break
        path, x = [], DST
        while prev[x]:
            path.append((prev[x], x)); x = prev[x]
        f = min(res[e] for e in path)
        for a, b in path:
            res[(a, b)] -= f; res[(b, a)] += f
        flow += f
    reach = set(prev)
    cut = [s for s, (u, v) in ARCS.items() if u in reach and v not in reach and cap[s] > 0]
    return flow, cut


def all_paths():
    """SRC -> DST 단순 경로(구간 이름 튜플) 전체."""
    out = []

    def dfs(node, segs, seen):
        if node == DST:
            out.append(tuple(segs)); return
        for s, (u, v) in ARCS.items():
            if u == node and v not in seen:
                dfs(v, segs + [s], seen | {v})
    dfs(SRC, [], {SRC})
    return out


PATHS = all_paths()


def assign(demand, closed=(), capmul=None, steps=40):
    """증분 배정: 수요를 steps조각으로 나눠, 전체 통과시간(대수 x 시간) 증가가 가장 작은 경로에 차례로 얹는다(시스템 최적 근사)."""
    cap = {s: CAP[s] * (capmul or {}).get(s, 1.0) for s in CAP}
    flow, pf = {s: 0.0 for s in CAP}, {}
    ok = [p for p in PATHS if not any(s in closed for s in p)]
    if not ok:
        return None
    for _ in range(steps):
        chunk = demand / steps
        best = min(ok, key=lambda p: sum((flow[s] + chunk) * seg_time(s, flow[s] + chunk, cap[s]) - flow[s] * seg_time(s, flow[s], cap[s]) for s in p))
        for s in best:
            flow[s] += chunk
        pf[best] = pf.get(best, 0) + chunk
    tt = {p: sum(seg_time(s, flow[s], cap[s]) for s in p) for p in pf}
    avg = sum(pf[p] * tt[p] for p in pf) / demand
    return {"flow": flow, "paths": pf, "path_min": tt, "avg_min": avg, "util": {s: flow[s] / cap[s] for s in cap if cap[s] > 0}}


FIXED_SHARE = {("A1", "A2", "A3"): 0.32, ("B1", "B3"): 0.40, ("C1", "C2"): 0.28}      # 과거 관행: 축선별 고정 배분(가상)


def fixed_assign(demand, closed=(), capmul=None):
    """과거 관행 배분: 축선별 고정 비율. 막힌 축선 물량은 열려 있는 축선에 비율대로 재분배."""
    cap = {s: CAP[s] * (capmul or {}).get(s, 1.0) for s in CAP}
    live = {p: w for p, w in FIXED_SHARE.items() if not any(s in closed for s in p)}
    if not live:
        return None
    tot = sum(live.values())
    flow = {s: 0.0 for s in CAP}
    pf = {p: demand * w / tot for p, w in live.items()}
    for p, f in pf.items():
        for s in p:
            flow[s] += f
    tt = {p: sum(seg_time(s, flow[s], cap[s]) for s in p) for p in pf}
    return {"flow": flow, "paths": pf, "path_min": tt, "avg_min": sum(pf[p] * tt[p] for p in pf) / demand,
            "util": {s: flow[s] / cap[s] for s in cap if cap[s] > 0}}


def lp_assign(demand, closed=(), capmul=None):
    """선형계획(LP): 구간 용량 상한(이용률 rho) 아래에서 자유주행시간 합을 최소화하는 경로별 물량. 불가능하면 rho를 1.0까지 완화."""
    cap = {s: CAP[s] * (capmul or {}).get(s, 1.0) for s in CAP}
    ok = [p for p in PATHS if not any(s in closed for s in p)]
    if not ok:
        return None
    for rho in (0.85, 0.95, 1.0):
        r = linprog([sum(T0[s] for s in p) for p in ok], A_ub=[[1.0 if s in p else 0.0 for p in ok] for s in CAP],
                    b_ub=[rho * cap[s] for s in CAP], A_eq=[[1.0] * len(ok)], b_eq=[demand], bounds=(0, None), method="highs")
        if r.status == 0:
            break
    else:
        return assign(demand, closed, capmul)                # 용량 초과 수요: 증분 배정으로 대체(과부하 표시)
    flow = {s: sum(r.x[i] for i, p in enumerate(ok) if s in p) for s in CAP}
    pf = {p: float(r.x[i]) for i, p in enumerate(ok) if r.x[i] > 1e-6}
    tt = {p: sum(seg_time(s, flow[s], cap[s]) for s in p) for p in pf}
    return {"flow": flow, "paths": pf, "path_min": tt, "avg_min": sum(pf[p] * tt[p] for p in pf) / demand,
            "util": {s: flow[s] / cap[s] for s in cap if cap[s] > 0}, "rho": rho}


def cost_of(res, tv=1000, fuel_km=450):
    """시간당 비용(원, 교육용 가정): 대당 시간비용 tv원/분 + 유류비 fuel_km원/km."""
    return sum(f * (res["path_min"][p] * tv + sum(KM[s] for s in p) * fuel_km) for p, f in res["paths"].items())
# ---------------------------------------------------------------------------

# 1) 최대유량과 병목(최소 절단)
f0, cut0 = max_flow()
print(f"[1] 노선망 최대 통과능력 {f0:.0f}대/시, 최소 절단(병목) 구간: {', '.join(cut0)}")
gain = []
for s in CAP:
    g, _ = max_flow(capmul={s: 1.2})
    gain.append((s, SEGS.set_index("seg").loc[s, "kind"], CAP[s], g - f0))
gain = pd.DataFrame(gain, columns=["구간", "유형", "용량", "용량 20% 증설 시 증가(대/시)"]).sort_values("용량 20% 증설 시 증가(대/시)", ascending=False)
print("[2] 구간별 증설 민감도(병목일수록 증설 효과가 큼)")
print(gain.head(5).to_string(index=False))

# 2) 사전 병목 식별: 계획 물량 기준 이용률 >= 0.9인 구간 + 최소 절단 구간
df = make_data(); d = add_features(df)
today = d[d.day == d.day.max()].set_index("seg")
risk = today[(today.util_eff >= 0.9)].index.tolist()
print(f"\n[3] 최종일(day {int(d.day.max())}) 사전 병목 경보: 유효 이용률 0.9 이상 구간 {risk or '없음'}")

# 3) 구간 폐쇄 시 우회: 최대유량 변화와 우회 경로
D = 200
print(f"\n[4] 구간 폐쇄 시나리오(수요 {D}대/시): 우회 경로와 평균 통과시간")
base = assign(D)
rows = [("평시", "-", f0, base["avg_min"], "A·B·C 분산", "수용")]
for c in (("A2",), ("B2",), ("A3",)):
    mf, cut = max_flow(c)
    r = assign(D, closed=c)
    top = max(r["paths"], key=r["paths"].get)
    rows.append((f"{c[0]} 폐쇄", ",".join(cut), mf, r["avg_min"], "->".join(top), "수용" if D <= mf else "과부하"))
sc = pd.DataFrame(rows, columns=["시나리오", "새 병목", "최대유량", "평균 통과(분)", "주 경로", "수요 대비"]).round(1)
print(sc.to_string(index=False))
print(f"    A2 폐쇄 시 평균 통과시간 +{rows[1][3] - rows[0][3]:.1f}분, 수요 {D}대/시 대비 최대유량 {rows[1][2]:.0f}대/시 -> {rows[1][5]}(과부하면 우회만으로 부족하므로 물량 조정·타 수단 검토가 필요)")

# 4) 파생 피처로 지체 예측 기준 모델: 시계열 교차검증(TimeSeriesSplit)
print(f"\n[5] 파생 피처 {len([f for f in FEATS if f not in ['plan_v','cap','cap_eff','t0','km','maint','rain']])}개 생성(lag·이용률·BPR·폭증비·요일 순환·구간 더미 등)")
days = np.sort(d.day.unique())
tscv = TimeSeriesSplit(n_splits=4)
res = []
for k, (a, b) in enumerate(tscv.split(days)):
    tr, te = d[d.day.isin(days[a])], d[d.day.isin(days[b])]
    m = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.08, random_state=0).fit(tr[FEATS], tr["delay"])
    p = te["t0"] * (1 + 0.08 * te["rain"]) + m.predict(te[FEATS])
    pb = te["t0"] + te["bpr_base"]
    res.append((k + 1, len(tr), len(te), mape(te["transit"], pb), mape(te["transit"], p)))
cv = pd.DataFrame(res, columns=["fold", "학습", "검증", "BPR MAPE(%)", "기준모델 MAPE(%)"]).round(2)
print(cv.to_string(index=False))
mm = cv["기준모델 MAPE(%)"].mean()
print(f"    교차검증 평균 MAPE {mm:.2f}% (목표 25% 이하: {'달성' if mm <= 25 else '미달'}), 폴드가 뒤로 갈수록 학습 구간이 길어져 미래 정보 누수가 없다")
m = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.08, random_state=0).fit(d[FEATS], d["delay"])
imp = pd.Series(m.feature_importances_, index=FEATS).sort_values(ascending=False)
print("    피처 중요도 상위 5:", ", ".join(f"{k} {v:.2f}" for k, v in imp.head(5).items()), f"/ 중요도 1% 미만 {int((imp < 0.01).sum())}개(차원 축소 후보)")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for s, (u, v) in ARCS.items():
    pos = {"SRC": (0, 1.5), "J1": (1, 3), "J2": (2, 3), "J3": (1.3, 1.5), "J4": (1, 0), "DST": (3, 1.5)}
    ax[0].plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], "r-" if s in cut0 else "gray", lw=2 + CAP[s] / 40)
    ax[0].text((pos[u][0] + pos[v][0]) / 2, (pos[u][1] + pos[v][1]) / 2, s, fontsize=9)
ax[0].set_title("Network (red = min-cut bottleneck)"); ax[0].axis("off")
imp.head(10)[::-1].plot.barh(ax=ax[1]); ax[1].set_title("Feature importance (top 10)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
[1] 노선망 최대 통과능력 290대/시, 최소 절단(병목) 구간: A2, B1, C1, X1
[2] 구간별 증설 민감도(병목일수록 증설 효과가 큼)
구간  유형  용량  용량 20% 증설 시 증가(대/시)
B1  일반 100                 15.0
C1  일반  70                 14.0
X1 교차로  50                 10.0
A2  교량  70                 10.0
A1  일반 130                  0.0

[3] 최종일(day 149) 사전 병목 경보: 유효 이용률 0.9 이상 구간 ['A1', 'A2']

[4] 구간 폐쇄 시나리오(수요 200대/시): 우회 경로와 평균 통과시간
 시나리오     새 병목  최대유량  평균 통과(분)     주 경로 수요 대비
   평시        - 290.0      40.9 A·B·C 분산    수용
A2 폐쇄 B1,C1,X1 220.0      44.9   B1->B3    수용
B2 폐쇄 A2,B3,C1 235.0      41.0   B1->B3    수용
A3 폐쇄    B3,C1 165.0      70.9   B1->B3   과부하
    A2 폐쇄 시 평균 통과시간 +4.0분, 수요 200대/시 대비 최대유량 220대/시 -> 수용(과부하면 우회만으로 부족하므로 물량 조정·타 수단 검토가 필요)

[5] 파생 피처 14개 생성(lag·이용률·BPR·폭증비·요일 순환·구간 더미 등)
 fold   학습  검증  BPR MAPE(%)  기준모델 MAPE(%)
    1  310 280         6.72          3.00
    2  590 280         6.02          2.54
    3  870 280         9.16          3.93
    4 1150 280         9.24          2.78
    교차검증 평균 MAPE 3.06% (목표 25% 이하: 달성), 폴드가 뒤로 갈수록 학습 구간이 길어져 미래 정보 누수가 없다
    피처 중요도 상위 5: util_eff 0.35, util_eff4 0.34, bpr_base 0.13, lag1_delay 0.09, km 0.02 / 중요도 1% 미만 14개(차원 축소 후보)
```
</details>
